# Electricity demand forecast

Section 19. Loads the saved model and scores one raw 10-minute record with `feature_pipeline.py`. History comes from the training file.


## 19. Inference check

Loads the saved models and scores the single raw record in `notebooks/sample_inference/inference_samples.csv`. That record uses the same columns as the training file. `feature_pipeline.py` attaches it to `Tetuan City power consumption.csv`, then runs the same cleaning, hourly aggregation, and feature engineering as sections 3–5 and 7. The forecast is for the hour after that record. `actual_target_t_plus_1` is filled only when that hour is already in the history, and it is not passed to `predict`. Run section 16.1 once after training. This cell can then run on its own.

In [ ]:
import pickle
import sys
import pandas as pd
from pathlib import Path

notebook_dir = Path("notebooks")
if not (notebook_dir / "feature_pipeline.py").exists():
    notebook_dir = Path(".")
sys.path.insert(0, str(notebook_dir.resolve()))
from feature_pipeline import forecast_frame_for_record

model_path = notebook_dir / "models" / "final_forecast_models.pkl"
csv_path = notebook_dir / "sample_inference" / "inference_samples.csv"
history_path = notebook_dir / "Tetuan City power consumption.csv"

with model_path.open("rb") as model_file:
    saved = pickle.load(model_file)
feature_columns = saved["feature_columns"]

record = pd.read_csv(csv_path)
history = pd.read_csv(history_path)
samples, built_feature_columns = forecast_frame_for_record(history, record)
if list(built_feature_columns) != list(feature_columns):
    raise ValueError("Built features do not match the saved model")
X = samples[feature_columns]

samples["pred_lightgbm"] = saved["lightgbm"].predict(X)
samples["pred_xgboost"] = saved["xgboost"].predict(X)
samples["pred_ensemble"] = (
    saved["weight_lightgbm"] * samples["pred_lightgbm"]
    + saved["weight_xgboost"] * samples["pred_xgboost"]
)

selected = saved["selected_model"]
pred_col = {
    "LightGBM": "pred_lightgbm",
    "XGBoost": "pred_xgboost",
    "Ensemble": "pred_ensemble",
}[selected]
samples["pred_selected"] = samples[pred_col]
samples["abs_error"] = (samples["pred_selected"] - samples["actual_target_t_plus_1"]).abs()

print(f"Loaded {model_path.resolve()}")
print(f"Selected model: {selected}")
print(f"Input record: {record['DateTime'].iloc[0]}")
display(samples[[
    "forecast_time",
    "pred_lightgbm",
    "pred_xgboost",
    "pred_ensemble",
    "pred_selected",
    "actual_target_t_plus_1",
    "abs_error",
]].round(1))


ModuleNotFoundError: No module named 'joblib'